# ViralScope AI — Complete Training Pipeline

**Architecture:** CLIP Vision Encoder + CLIP Text Encoder → Cross-Modal Fusion MLP → Binary viral classifier  
**Hardware targets:** GTX 1650 4 GB (config defaults) · Google Colab T4 15 GB (auto-scaled)  
**Data:** 10 000 YouTube videos — thumbnails + titles — US / GB / CA trending datasets  

| Cell | Step |
|------|------|
| 1 | Setup: detect runtime, set working dir |
| 2 | Load & validate `config.yaml` — single source of truth |
| 3 | Verify CSV files |
| 4 | Load, combine & clean data |
| 5 | Compute LOO viral labels, sample to 10 000 |
| 6 | Download thumbnails (parallel) |
| 7 | Tokenize titles with CLIP tokenizer, save tensors |
| 8 | Stratified train / val / test splits |
| 9 | Model architecture definition |
| 10 | Dataset class & DataLoaders |
| 11 | Loss function & training utilities |
| 12 | Initialise model, optimizer, scheduler |
| 13 | Two-stage training loop |
| 14 | Evaluate on test set |
| 15 | Save best model & training log |

> **Estimated runtime:** ~3–5 h on GTX 1650 (50 epochs, batch 4) · ~60–90 min on Colab T4 (batch 16)


In [1]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/ViralScope-AI'):
        import subprocess
        subprocess.run(['git','clone','https://github.com/yassine-cmd/ViralScope-AI.git',
                        '/content/ViralScope-AI'], check=True)
    sys.path.insert(0, '/content/ViralScope-AI')
    os.chdir('/content/ViralScope-AI')
else:
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f'[Setup] Runtime     : {"Google Colab" if IN_COLAB else "Local"}')
print(f'[Setup] Working dir : {os.getcwd()}')


[Setup] Runtime     : Local
[Setup] Working dir : d:\work\cycle\S4\NLP\ProjectNLP


In [2]:
import yaml, torch

# ── Load ────────────────────────────────────────────────────────────────────
_CONFIG_PATH = 'config.yaml'
if not os.path.exists(_CONFIG_PATH):
    raise FileNotFoundError(
        f'{_CONFIG_PATH} not found in {os.getcwd()}. '
        'Place config.yaml in the working directory before continuing.'
    )
with open(_CONFIG_PATH) as _f:
    CONFIG = yaml.safe_load(_f)
if not CONFIG:
    raise ValueError('config.yaml is empty — provide a valid configuration file.')

# ── Strict key validation — every key used in the notebook must exist ────────
_REQUIRED = {
    'project':           ['name','seed','device'],
    'data':              ['raw_dir','processed_dir','tensor_dir','splits_dir',
                          'train_split','val_split','test_split',
                          'min_dataset_size','target_threshold',
                          'thumbnail_url_template','thumbnail_fallback_url',
                          'thumbnail_download_workers'],
    'training':          ['batch_size','num_workers','epochs',
                          'lr_head','lr_head_stage2','lr_backbone_cv','lr_backbone_nlp',
                          'weight_decay','betas',
                          'loss_function','focal_loss_gamma',
                          'scheduler','scheduler_T_0','scheduler_T_mult','scheduler_eta_min',
                          'scheduler_factor','scheduler_patience',
                          'early_stopping_patience','gradient_clip_norm','two_stage'],
    'training.two_stage':['enabled','head_warmup_epochs','unfreeze_cv','unfreeze_nlp'],
    'model':             ['clip','fusion'],
    'model.clip':        ['checkpoint','feature_dim','max_seq_length','freeze_backbone'],
    'model.fusion':      ['hidden_layers','dropout','activation'],
    'augmentation':      ['resize_size','rotation_range',
                          'color_jitter_brightness','color_jitter_contrast','color_jitter_saturation',
                          'horizontal_flip_prob','normalize_mean','normalize_std'],
    'paths':             ['checkpoints','best_model','training_log','results'],
}
_missing = []
for _sec, _keys in _REQUIRED.items():
    if '.' in _sec:
        _top, _sub = _sec.split('.',1)
        _node = CONFIG.get(_top,{}).get(_sub)
        if _node is None:
            _missing.append(f"  CONFIG['{_top}']['{_sub}'] block is missing")
            continue
        for _k in _keys:
            if _k not in _node:
                _missing.append(f"  CONFIG['{_top}']['{_sub}']['{_k}'] is missing")
    else:
        if _sec not in CONFIG:
            _missing.append(f"  CONFIG['{_sec}'] block is missing")
            continue
        for _k in _keys:
            if _k not in CONFIG[_sec]:
                _missing.append(f"  CONFIG['{_sec}']['{_k}'] is missing")
if _missing:
    raise KeyError('Config validation failed — missing required keys:\n' + '\n'.join(_missing))

# ── Create required directories ──────────────────────────────────────────────
for _d in [
    CONFIG['data']['raw_dir'],
    f"{CONFIG['data']['raw_dir']}/thumbnails",
    CONFIG['data']['processed_dir'],
    CONFIG['data']['tensor_dir'],
    CONFIG['data']['splits_dir'],
    CONFIG['paths']['checkpoints'],
    CONFIG['paths']['results'],
    os.path.dirname(CONFIG['paths']['best_model']),
    os.path.dirname(CONFIG['paths']['training_log']),
]:
    os.makedirs(_d, exist_ok=True)

# ── Hardware-aware runtime overrides ─────────────────────────────────────────
# config.yaml stores conservative GTX-1650 values.
# On Colab T4 (>=12 GB) we scale up batch_size and num_workers automatically.
# RUNTIME_ variables are used in all subsequent cells instead of CONFIG directly
# for batch_size / num_workers so the override is explicit and logged.
_vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9
            if torch.cuda.is_available() else 0)
if IN_COLAB and _vram_gb >= 12:
    RUNTIME_BATCH_SIZE  = 16
    RUNTIME_NUM_WORKERS = 4
    print(f'[Config] Colab GPU detected ({_vram_gb:.1f} GB) — overriding: '
          f'batch_size={RUNTIME_BATCH_SIZE}, num_workers={RUNTIME_NUM_WORKERS}')
else:
    RUNTIME_BATCH_SIZE  = CONFIG['training']['batch_size']
    RUNTIME_NUM_WORKERS = CONFIG['training']['num_workers']
    print(f'[Config] Using config.yaml values: '
          f'batch_size={RUNTIME_BATCH_SIZE}, num_workers={RUNTIME_NUM_WORKERS}')

failed_video_ids = []   # populated in Cell 6; declared here so all cells can reference it

print(f'[Config] Loaded and validated : {_CONFIG_PATH}')
print(f'[Config] Project              : {CONFIG["project"]["name"]}')
print(f'[Config] Seed                 : {CONFIG["project"]["seed"]}')
print(f'[Config] Dataset size limit   : {CONFIG["data"]["min_dataset_size"]:,}')


[Config] Using config.yaml values: batch_size=4, num_workers=0
[Config] Loaded and validated : config.yaml
[Config] Project              : ViralScope AI
[Config] Seed                 : 42
[Config] Dataset size limit   : 10,000


In [4]:
_raw_dir   = CONFIG['data']['raw_dir']
_csv_files = [f for f in os.listdir(_raw_dir) if f.endswith('.csv') and f != 'trending.csv']

if not _csv_files:
    print(f'[Data] No CSV files found in {_raw_dir}.')
    print('[Data] Expected: USvideos.csv, GBvideos.csv, CAvideos.csv')
    if IN_COLAB:
        from google.colab import files as _cf
        _uploaded = _cf.upload()
        for _fname, _content in _uploaded.items():
            with open(os.path.join(_raw_dir, _fname), 'wb') as _fh:
                _fh.write(_content)
            print(f'[Data] Saved: {_fname}')
        _csv_files = [f for f in os.listdir(_raw_dir) if f.endswith('.csv') and f != 'trending.csv']
    else:
        raise FileNotFoundError(
            f'Place the CSV files in {os.path.abspath(_raw_dir)} and re-run this cell.')

if not _csv_files:
    raise ValueError('[Data] No CSV files available after upload. Cannot continue.')

_ENGLISH_MARKETS = {'US','GB','CA'}
_detected = {f.replace('videos.csv','').replace('videos','') for f in _csv_files}
_non_en   = _detected - _ENGLISH_MARKETS
if _non_en:
    print(f'[Data] WARNING: non-English market files detected: {_non_en}')
    print('[Data]          CLIP text encoder is English-only; accuracy may degrade.')

print(f'[Data] Found {len(_csv_files)} CSV file(s): {_csv_files}')


[Data] Found 3 CSV file(s): ['CAvideos.csv', 'GBvideos.csv', 'USvideos.csv']


In [5]:
import pandas as pd, numpy as np

_raw_dir       = CONFIG['data']['raw_dir']
_processed_dir = CONFIG['data']['processed_dir']
_csv_files     = [f for f in os.listdir(_raw_dir) if f.endswith('.csv') and f != 'trending.csv']

print(f'[Data] Loading {len(_csv_files)} CSV file(s)...')
_all_dfs = []
for _csv_file in _csv_files:
    try:
        _df = pd.read_csv(os.path.join(_raw_dir, _csv_file),
                          on_bad_lines='skip', low_memory=False, encoding='utf-8')
        if 'video_id' not in _df.columns and 'id' in _df.columns:
            _df = _df.rename(columns={'id': 'video_id'})
        _all_dfs.append(_df)
        print(f'[Data]   {_csv_file}: {len(_df):,} rows')
    except Exception as _e:
        print(f'[Data]   ERROR loading {_csv_file}: {_e}')

if not _all_dfs:
    raise ValueError('[Data] No data loaded — verify CSV files are valid and non-empty.')

_trending_df = pd.concat(_all_dfs, ignore_index=True)
_trending_df.to_csv(f'{_raw_dir}/trending.csv', index=False)
print(f'[Data] Combined total : {len(_trending_df):,} rows  ->  saved trending.csv')

# ── Clean ───────────────────────────────────────────────────────────────────
def clean_dataset(df):
    """Deduplicate, drop invalid rows, retain required columns."""
    if 'video_id' in df.columns:
        df = df.dropna(subset=['video_id'])
        df = df[df['video_id'].astype(str).str.strip() != '']
    if 'views' in df.columns:
        df['views'] = pd.to_numeric(df['views'], errors='coerce')
        df = df.dropna(subset=['views'])
        df = df[df['views'] > 0]
    if 'channel_id' not in df.columns:
        df['channel_id'] = df['channel_title'].astype(str) if 'channel_title' in df.columns else 'unknown'
    if 'title' not in df.columns:
        df['title'] = 'Untitled'
    else:
        df['title'] = df['title'].fillna('Untitled').astype(str)
        df = df[df['title'].str.strip() != '']
    if 'video_id' in df.columns:
        df = df.drop_duplicates(subset=['video_id'], keep='last')
    _keep = ['video_id','title','views','channel_title','channel_id']
    return df[[c for c in _keep if c in df.columns]].reset_index(drop=True)

clean_df = clean_dataset(_trending_df)
clean_df.to_csv(f'{_processed_dir}/clean_dataset.csv', index=False)
print(f'[Data] After cleaning : {len(clean_df):,} rows  ->  saved clean_dataset.csv')


[Data] Loading 3 CSV file(s)...
[Data]   CAvideos.csv: 40,881 rows
[Data]   GBvideos.csv: 38,916 rows
[Data]   USvideos.csv: 40,949 rows
[Data] Combined total : 120,746 rows  ->  saved trending.csv
[Data] After cleaning : 30,318 rows  ->  saved clean_dataset.csv


In [6]:
def compute_labels(config, df):
    """Leave-One-Out viral labelling: viral if views > threshold * channel LOO mean."""
    _stats = df.groupby('channel_id')['views'].agg(['sum','count','mean']).reset_index()
    _stats.columns = ['channel_id','channel_sum','video_count','channel_avg']
    df = df.merge(_stats, on='channel_id', how='left')
    df['channel_loo_avg_views'] = (
        (df['channel_sum'] - df['views']) / (df['video_count'] - 1).clip(lower=1)
    )
    df['multiplier'] = df['views'] / (df['channel_loo_avg_views'] + 1e-5)
    _reliable = _stats[_stats['video_count'] >= 2]['channel_id'].tolist()
    df = df[df['channel_id'].isin(_reliable)]
    _threshold = config['data']['target_threshold']
    df['is_viral'] = (df['multiplier'] > _threshold).astype(int)
    _minority_pct = min(df['is_viral'].mean(), 1 - df['is_viral'].mean())
    if _minority_pct < 0.10:
        _threshold = df['multiplier'].quantile(0.75)
        df['is_viral'] = (df['multiplier'] > _threshold).astype(int)
        print(f'[Labels] Class imbalance < 10% — dynamic threshold: {_threshold:.2f}')
    _cols = ['video_id','title','views','channel_loo_avg_views','multiplier','is_viral','channel_id']
    return df[[c for c in _cols if c in df.columns]].reset_index(drop=True)

labeled_df = compute_labels(CONFIG, clean_df)
print(f'[Labels] Viral: {labeled_df["is_viral"].sum():,}  |  '
      f'Non-viral: {(labeled_df["is_viral"]==0).sum():,}')

# Cap dataset size — value comes directly from config, no default fallback
_limit = CONFIG['data']['min_dataset_size']
if len(labeled_df) > _limit:
    labeled_df = (
        labeled_df
        .sample(n=_limit, random_state=CONFIG['project']['seed'])
        .reset_index(drop=True)
    )
    print(f'[Labels] Sampled to {_limit:,} rows  (config: data.min_dataset_size)')

labeled_df.to_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv", index=False)
print(f'[Labels] Saved labeled_dataset.csv — {len(labeled_df):,} rows')
print(f'[Labels] Class balance — viral: {labeled_df["is_viral"].mean()*100:.1f}%  '
      f'non-viral: {(1-labeled_df["is_viral"].mean())*100:.1f}%')


[Labels] Viral: 4,815  |  Non-viral: 21,451
[Labels] Sampled to 10,000 rows  (config: data.min_dataset_size)
[Labels] Saved labeled_dataset.csv — 10,000 rows
[Labels] Class balance — viral: 18.6%  non-viral: 81.4%


In [ ]:
import requests
from tqdm.auto import tqdm
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_thumbnails(config, df):
    """Download thumbnails in parallel from YouTube CDN. Returns list of failed video_ids."""
    _thumb_dir = f"{config['data']['raw_dir']}/thumbnails"
    os.makedirs(_thumb_dir, exist_ok=True)
    _template  = config['data']['thumbnail_url_template']
    _fallback  = config['data']['thumbnail_fallback_url']
    _workers   = config['data']['thumbnail_download_workers']
    _video_ids = df['video_id'].tolist()

    def _download_one(vid):
        _path = os.path.join(_thumb_dir, f'{vid}.jpg')
        if os.path.exists(_path):
            return vid, 'exists'
        try:
            _r = requests.get(_template.format(video_id=vid), timeout=5)
            if _r.status_code != 200:
                _r = requests.get(_fallback.format(video_id=vid), timeout=5)
            if _r.status_code == 200:
                _img = Image.open(BytesIO(_r.content))
                if _img.size[0] < 120 or _img.size[1] < 80:
                    return vid, False   # discard tiny placeholder
                _img.save(_path)
                return vid, True
        except Exception:
            pass
        return vid, False

    _ok, _fail, _skip = 0, 0, 0
    _failed_ids = []
    with ThreadPoolExecutor(max_workers=_workers) as _pool:
        _futures = {_pool.submit(_download_one, v): v for v in _video_ids}
        for _fut in tqdm(as_completed(_futures), total=len(_video_ids), desc='Thumbnails'):
            _vid, _status = _fut.result()
            if   _status is True:     _ok   += 1
            elif _status is False:    _fail += 1; _failed_ids.append(_vid)
            else:                     _skip += 1

    print(f'[Thumbnails] Downloaded: {_ok:,}  |  Failed: {_fail:,}  |  Skipped (cached): {_skip:,}')
    if _fail > 0:
        print(f'[Thumbnails] {_fail} video(s) will be excluded from training.')
    return _failed_ids

failed_video_ids = download_thumbnails(CONFIG, labeled_df)


d:\work\cycle\S4\NLP\ProjectNLP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Thumbnails:  71%|███████   | 7053/10000 [20:16<07:59,  6.14it/s] 

In [11]:
import torch
from transformers import CLIPTokenizer

def tokenize_titles(config, df):
    """Tokenize all titles once and persist to disk for fast DataLoader reuse."""
    _tensor_dir = config['data']['tensor_dir']
    _checkpoint = config['model']['clip']['checkpoint']
    _max_len    = config['model']['clip']['max_seq_length']
    _titles     = df['title'].fillna('Untitled').astype(str).tolist()

    print(f'[Tokenize] Checkpoint : {_checkpoint}')
    print(f'[Tokenize] max_length : {_max_len}  |  titles: {len(_titles):,}')

    _tok = CLIPTokenizer.from_pretrained(_checkpoint)
    _enc = _tok(_titles, max_length=_max_len, padding='max_length',
                truncation=True, return_tensors='pt')
    _ids   = _enc['input_ids']
    _masks = _enc['attention_mask']

    torch.save(_ids,   f'{_tensor_dir}/input_ids.pt')
    torch.save(_masks, f'{_tensor_dir}/attention_masks.pt')
    print(f'[Tokenize] Tensors saved — shape: {tuple(_ids.shape)}')
    return _ids, _masks

# Remove videos whose thumbnail download failed before tokenizing
labeled_df = labeled_df[~labeled_df['video_id'].isin(failed_video_ids)].reset_index(drop=True)
print(f'[Tokenize] Rows after removing {len(failed_video_ids)} failed thumbnails: {len(labeled_df):,}')

input_ids, attention_masks = tokenize_titles(CONFIG, labeled_df)


[Tokenize] Rows after removing 2143 failed thumbnails: 7,857
[Tokenize] Checkpoint : openai/clip-vit-base-patch32
[Tokenize] max_length : 77  |  titles: 7,857
[Tokenize] Tensors saved — shape: (7857, 77)


In [12]:
from sklearn.model_selection import train_test_split

def create_splits(config, df):
    _idx        = np.arange(len(df))
    _tr         = config['data']['train_split']
    _va         = config['data']['val_split']
    _te         = config['data']['test_split']
    _seed       = config['project']['seed']
    assert abs(_tr + _va + _te - 1.0) < 1e-6, (
        f'Split fractions must sum to 1.0, got {_tr+_va+_te}')
    _train_idx, _tmp = train_test_split(
        _idx, train_size=_tr, random_state=_seed, stratify=df['is_viral'])
    _val_idx, _test_idx = train_test_split(
        _tmp, train_size=_va/(_va+_te), random_state=_seed,
        stratify=df.iloc[_tmp]['is_viral'])
    print(f'[Splits] Train: {len(_train_idx):,}  |  Val: {len(_val_idx):,}  |  Test: {len(_test_idx):,}')
    return _train_idx, _val_idx, _test_idx

train_idx, val_idx, test_idx = create_splits(CONFIG, labeled_df)


[Splits] Train: 5,499  |  Val: 1,179  |  Test: 1,179


In [13]:
import torch.nn as nn
from transformers import CLIPModel


class CVExtractor(nn.Module):
    """CLIP Vision Encoder  ->  512-d L2-normalised image embedding."""
    def __init__(self, vision_model, visual_projection, freeze=True):
        super().__init__()
        self.vision_model      = vision_model
        self.visual_projection = visual_projection
        if freeze:
            for p in self.parameters(): p.requires_grad = False

    def forward(self, pixel_values):
        _out  = self.vision_model(pixel_values=pixel_values)
        _proj = self.visual_projection(_out.pooler_output)   # (B, 768) -> (B, 512)
        return nn.functional.normalize(_proj, p=2, dim=1)


class NLPExtractor(nn.Module):
    """CLIP Text Encoder  ->  512-d L2-normalised title embedding."""
    def __init__(self, text_model, text_projection, freeze=True):
        super().__init__()
        self.text_model      = text_model
        self.text_projection = text_projection
        if freeze:
            for p in self.parameters(): p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        _out  = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        _proj = self.text_projection(_out.pooler_output)     # (B, 512)
        return nn.functional.normalize(_proj, p=2, dim=1)


class FusionMLP(nn.Module):
    """
    Cross-modal fusion head.
    Concatenates: [img | txt | |img-txt| | img*txt | cos_sim]  =  4*512+1 = 2049 dims.
    Projects down through hidden layers to a single logit.
    """
    def __init__(self, feature_dim, hidden_layers, dropout, activation):
        super().__init__()
        _act_cls  = getattr(nn, activation)
        _in_dim   = 4 * feature_dim + 1
        _layers   = []
        for _h in hidden_layers:
            _layers += [nn.Linear(_in_dim,_h), nn.LayerNorm(_h), _act_cls(), nn.Dropout(dropout)]
            _in_dim  = _h
        _layers.append(nn.Linear(_in_dim, 1))
        self.network = nn.Sequential(*_layers)

    def forward(self, img_emb, txt_emb):
        _cos  = (img_emb * txt_emb).sum(dim=1, keepdim=True)   # (B,1)
        _diff = torch.abs(img_emb - txt_emb)                    # (B,512)
        _prod = img_emb * txt_emb                               # (B,512)
        return self.network(torch.cat([img_emb, txt_emb, _diff, _prod, _cos], dim=1)).squeeze(-1)


class ViralScopeModel(nn.Module):
    """Full multi-modal classifier — all hyperparameters driven by config.yaml."""
    def __init__(self, config):
        super().__init__()
        _cc  = config['model']['clip']
        _fc  = config['model']['fusion']
        _clip = CLIPModel.from_pretrained(_cc['checkpoint'])
        self.cv_extractor  = CVExtractor(_clip.vision_model, _clip.visual_projection, _cc['freeze_backbone'])
        self.nlp_extractor = NLPExtractor(_clip.text_model,  _clip.text_projection,   _cc['freeze_backbone'])
        self.fusion        = FusionMLP(_cc['feature_dim'], _fc['hidden_layers'], _fc['dropout'], _fc['activation'])

    def forward(self, images, input_ids, attention_mask):
        return self.fusion(self.cv_extractor(images), self.nlp_extractor(input_ids, attention_mask))

    def predict_proba(self, images, input_ids, attention_mask):
        return torch.sigmoid(self.forward(images, input_ids, attention_mask))


_fd = CONFIG['model']['clip']['feature_dim']
print(f'[Model] CVExtractor  : CLIP ViT-B/32 vision  ->  {_fd}-d embedding')
print(f'[Model] NLPExtractor : CLIP ViT-B/32 text    ->  {_fd}-d embedding')
print(f'[Model] FusionMLP    : {4*_fd+1} -> {CONFIG["model"]["fusion"]["hidden_layers"]} -> 1 (logit)')
print(f'[Model] Stage-1 backbone frozen: {CONFIG["model"]["clip"]["freeze_backbone"]}')


[Model] CVExtractor  : CLIP ViT-B/32 vision  ->  512-d embedding
[Model] NLPExtractor : CLIP ViT-B/32 text    ->  512-d embedding
[Model] FusionMLP    : 2049 -> [256, 64] -> 1 (logit)
[Model] Stage-1 backbone frozen: True


In [14]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, UnidentifiedImageError

# ── Reload tensors & labeled CSV (safe if cell is re-run independently) ───────
input_ids       = torch.load(f"{CONFIG['data']['tensor_dir']}/input_ids.pt",       weights_only=True)
attention_masks = torch.load(f"{CONFIG['data']['tensor_dir']}/attention_masks.pt", weights_only=True)
labeled_df      = pd.read_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv")
labeled_df      = labeled_df[~labeled_df['video_id'].isin(failed_video_ids)].reset_index(drop=True)
print(f'[DataLoader] Tensors reloaded: {tuple(input_ids.shape)}  |  rows: {len(labeled_df):,}')

# ── Image transforms — all values from config ────────────────────────────────
_aug  = CONFIG['augmentation']
_px   = _aug['resize_size'][0]
_mean = _aug['normalize_mean']
_std  = _aug['normalize_std']
_BICUBIC = transforms.InterpolationMode.BICUBIC

train_transform = transforms.Compose([
    transforms.Resize(_px, interpolation=_BICUBIC),
    transforms.CenterCrop(_px),
    transforms.RandomHorizontalFlip(p=_aug['horizontal_flip_prob']),
    transforms.RandomRotation(degrees=_aug['rotation_range'][1]),
    transforms.ColorJitter(
        brightness=_aug['color_jitter_brightness'],
        contrast=_aug['color_jitter_contrast'],
        saturation=_aug['color_jitter_saturation']),
    transforms.ToTensor(),
    transforms.Normalize(_mean, _std),
])
eval_transform = transforms.Compose([
    transforms.Resize(_px, interpolation=_BICUBIC),
    transforms.CenterCrop(_px),
    transforms.ToTensor(),
    transforms.Normalize(_mean, _std),
])

# ── Dataset ──────────────────────────────────────────────────────────────────
class ViralScopeDataset(Dataset):
    def __init__(self, indices, df, transform, ids, masks):
        self.df        = df.iloc[indices].reset_index(drop=True)
        self.transform = transform
        self.ids       = ids[indices]
        self.masks     = masks[indices]
        self.thumb_dir = f"{CONFIG['data']['raw_dir']}/thumbnails"

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        _row  = self.df.iloc[idx]
        _path = os.path.join(self.thumb_dir, f"{_row['video_id']}.jpg")
        try:
            _img = Image.open(_path).convert('RGB')
        except (FileNotFoundError, OSError, UnidentifiedImageError):
            _img = Image.new('RGB', (_px, _px), (128, 128, 128))  # grey placeholder
        return {
            'image':          self.transform(_img),
            'input_ids':      self.ids[idx],
            'attention_mask': self.masks[idx],
            'label':          torch.tensor(_row['is_viral'], dtype=torch.float32),
        }

train_ds = ViralScopeDataset(train_idx, labeled_df, train_transform, input_ids, attention_masks)
val_ds   = ViralScopeDataset(val_idx,   labeled_df, eval_transform,  input_ids, attention_masks)
test_ds  = ViralScopeDataset(test_idx,  labeled_df, eval_transform,  input_ids, attention_masks)

# ── WeightedRandomSampler (balances class imbalance without modifying loss) ──
_tl       = labeled_df.iloc[train_idx]['is_viral'].values
_cc       = np.bincount(_tl)
_sw       = torch.tensor([1.0/_cc[int(t)] for t in _tl], dtype=torch.double)
sampler   = WeightedRandomSampler(_sw, len(_sw), replacement=True)

# ── DataLoaders  (RUNTIME_ vars set in Cell 2 based on detected hardware) ────
_pf  = 4    if RUNTIME_NUM_WORKERS > 0 else None
_pw  = True if RUNTIME_NUM_WORKERS > 0 else False
_pin = torch.cuda.is_available()
_dl_kwargs = dict(num_workers=RUNTIME_NUM_WORKERS, pin_memory=_pin,
                  prefetch_factor=_pf, persistent_workers=_pw)

train_loader = DataLoader(train_ds, batch_size=RUNTIME_BATCH_SIZE, sampler=sampler,  **_dl_kwargs)
val_loader   = DataLoader(val_ds,   batch_size=RUNTIME_BATCH_SIZE, shuffle=False,    **_dl_kwargs)
test_loader  = DataLoader(test_ds,  batch_size=RUNTIME_BATCH_SIZE, shuffle=False,    **_dl_kwargs)

print(f'[DataLoader] Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
print(f'[DataLoader] batch_size={RUNTIME_BATCH_SIZE}, num_workers={RUNTIME_NUM_WORKERS}')


[DataLoader] Tensors reloaded: (7857, 77)  |  rows: 7,857
[DataLoader] Train: 5,499  Val: 1,179  Test: 1,179
[DataLoader] batch_size=4, num_workers=0


In [15]:
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

class FocalLoss(nn.Module):
    """Binary focal loss with optional per-class alpha weighting."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction

    def forward(self, logits, targets):
        _p_t    = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
        _fl_w   = (1 - _p_t) ** self.gamma
        _ce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        if self.alpha is not None:
            _fl_w = _fl_w * (self.alpha[1]*targets + self.alpha[0]*(1-targets))
        _loss   = _fl_w * _ce
        return _loss.mean() if self.reduction == 'mean' else _loss


def build_criterion(config, dataset):
    """Build loss from config. Supports 'BCEWithLogitsLoss' or 'FocalLoss'."""
    _labels    = dataset.df['is_viral'].values
    _loss_type = config['training']['loss_function']      # required key — no default
    if _loss_type == 'FocalLoss':
        _alpha_cfg = config['training'].get('focal_loss_alpha')  # optional key
        if _alpha_cfg is None:
            _n_pos    = (_labels == 1).sum()
            _n_neg    = (_labels == 0).sum()
            _total    = _n_pos + _n_neg
            _alpha    = torch.tensor([_total/(2*_n_neg), _total/(2*_n_pos)], dtype=torch.float32)
            print(f'[Loss] FocalLoss — auto alpha: {_alpha.tolist()}')
        else:
            _ap    = float(_alpha_cfg)
            _alpha = torch.tensor([1.0-_ap, _ap], dtype=torch.float32)
            print(f'[Loss] FocalLoss — config alpha: {_alpha.tolist()}')
        return FocalLoss(alpha=_alpha, gamma=config['training']['focal_loss_gamma'])
    elif _loss_type == 'BCEWithLogitsLoss':
        print('[Loss] BCEWithLogitsLoss (class imbalance handled by WeightedRandomSampler)')
        return nn.BCEWithLogitsLoss()
    else:
        raise ValueError(f"[Loss] Unsupported loss_function in config: '{_loss_type}'. "
                         "Expected 'BCEWithLogitsLoss' or 'FocalLoss'.")


def compute_metrics(preds, targets, threshold=0.5):
    _pb = (preds >= threshold).float()
    _tp = ((_pb==1) & (targets==1)).sum().item()
    _tn = ((_pb==0) & (targets==0)).sum().item()
    _fp = ((_pb==1) & (targets==0)).sum().item()
    _fn = ((_pb==0) & (targets==1)).sum().item()
    _p  = _tp / (_tp + _fp + 1e-10)
    _r  = _tp / (_tp + _fn + 1e-10)
    return {
        'accuracy':  (_tp + _tn) / (_tp + _tn + _fp + _fn + 1e-10),
        'precision': _p,
        'recall':    _r,
        'f1':        2*_p*_r / (_p + _r + 1e-10),
        'auc_roc':   roc_auc_score(targets.numpy(), preds.numpy()),
        'pr_auc':    average_precision_score(targets.numpy(), preds.numpy()),
    }


def _amp_ctx(device):
    """Return AMP autocast context — enabled only for CUDA."""
    if device.type == 'cuda':
        return torch.amp.autocast('cuda')
    import contextlib
    return contextlib.nullcontext()


def train_epoch(model, loader, criterion, optimizer, scaler, device, clip_norm):
    """Train one epoch with mixed-precision and gradient clipping."""
    model.train()
    _total_loss, _preds, _labels = 0.0, [], []
    for _batch in loader:
        _imgs  = _batch['image'].to(device, non_blocking=True)
        _ids   = _batch['input_ids'].to(device, non_blocking=True)
        _masks = _batch['attention_mask'].to(device, non_blocking=True)
        _lbl   = _batch['label'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with _amp_ctx(device):
            _logits = model(_imgs, _ids, _masks)
            _loss   = criterion(_logits, _lbl)
        if scaler is not None:
            scaler.scale(_loss).backward()
            scaler.unscale_(optimizer)
            if clip_norm > 0: torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            _loss.backward()
            if clip_norm > 0: torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()
        _total_loss += _loss.item()
        _preds.extend(torch.sigmoid(_logits).detach().cpu())
        _labels.extend(_lbl.cpu())
    _p = torch.stack(_preds); _l = torch.stack(_labels)
    _m = compute_metrics(_p, _l); _m['loss'] = _total_loss / len(loader)
    return _m


@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    """Validate one epoch with mixed-precision."""
    model.eval()
    _total_loss, _preds, _labels = 0.0, [], []
    for _batch in loader:
        _imgs  = _batch['image'].to(device, non_blocking=True)
        _ids   = _batch['input_ids'].to(device, non_blocking=True)
        _masks = _batch['attention_mask'].to(device, non_blocking=True)
        _lbl   = _batch['label'].to(device, non_blocking=True)
        with _amp_ctx(device):
            _logits = model(_imgs, _ids, _masks)
            _loss   = criterion(_logits, _lbl)
        _total_loss += _loss.item()
        _preds.extend(torch.sigmoid(_logits).cpu())
        _labels.extend(_lbl.cpu())
    _p = torch.stack(_preds); _l = torch.stack(_labels)
    _m = compute_metrics(_p, _l); _m['loss'] = _total_loss / len(loader)
    return _m


def unfreeze_backbone(model_raw, name, lr, optimizer):
    """Unfreeze a named backbone and register its parameters with the optimizer."""
    _bb = getattr(model_raw, name)
    for _p in _bb.parameters(): _p.requires_grad = True
    optimizer.add_param_group({
        'params': [p for p in _bb.parameters() if p.requires_grad],
        'lr': lr,
    })
    _n = sum(p.numel() for p in _bb.parameters() if p.requires_grad)
    print(f'[Train]   Unfroze {name}: {_n:,} params  lr={lr}')


def build_scheduler(optimizer, config):
    """Build LR scheduler from config — used for both Stage 1 and Stage 2 resets."""
    _cfg = config['training']
    _sched_name = _cfg['scheduler']
    if _sched_name == 'CosineAnnealingWarmRestarts':
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=_cfg['scheduler_T_0'],
            T_mult=_cfg['scheduler_T_mult'],
            eta_min=_cfg['scheduler_eta_min'],
        )
    elif _sched_name == 'ReduceLROnPlateau':
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max',
            factor=_cfg['scheduler_factor'],
            patience=_cfg['scheduler_patience'],
        )
    else:
        raise ValueError(f"[Scheduler] Unsupported scheduler in config: '{_sched_name}'. "
                         "Expected 'CosineAnnealingWarmRestarts' or 'ReduceLROnPlateau'.")


print('[Utils] Loss, metrics, train/val epoch functions, scheduler builder defined.')


[Utils] Loss, metrics, train/val epoch functions, scheduler builder defined.


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[Init] Device : {device}')
if torch.cuda.is_available():
    print(f'[Init] GPU    : {torch.cuda.get_device_name(0)}')
    print(f'[Init] VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# model_raw is kept for backbone attribute access during Stage 2 unfreeze.
# torch.compile() wraps submodules, hiding named attributes like cv_extractor.
model_raw = ViralScopeModel(CONFIG).to(device)

_cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
if _cc >= 8:
    print(f'[Init] Compute capability {_cc}.x — applying torch.compile()')
    model = torch.compile(model_raw)
else:
    print(f'[Init] Compute capability {_cc}.x — skipping torch.compile() (requires CC >= 8.0)')
    model = model_raw

_tcfg = CONFIG['training']

# Stage 1: only fusion head params (backbones frozen)
_head_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(
    _head_params,
    lr=_tcfg['lr_head'],
    weight_decay=_tcfg['weight_decay'],
    betas=tuple(_tcfg['betas']),    # required key — no default
)

# GradScaler only on CUDA
scaler    = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

criterion = build_criterion(CONFIG, train_ds)
scheduler = build_scheduler(optimizer, CONFIG)

_n_head = sum(p.numel() for p in _head_params)
_n_all  = sum(p.numel() for p in model.parameters())
print(f'[Init] Trainable params (Stage 1 head only): {_n_head:,}')
print(f'[Init] Total params                        : {_n_all:,}')
print(f'[Init] Scheduler                           : {_tcfg["scheduler"]}')


[Init] Device : cuda
[Init] GPU    : NVIDIA GeForce GTX 1650
[Init] VRAM   : 4.3 GB


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 12544.30it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Init] Compute capability 7.x — skipping torch.compile() (requires CC >= 8.0)
[Loss] BCEWithLogitsLoss (class imbalance handled by WeightedRandomSampler)
[Init] Trainable params (Stage 1 head only): 541,953
[Init] Total params                        : 151,819,265
[Init] Scheduler                           : CosineAnnealingWarmRestarts


In [ ]:
_tcfg             = CONFIG['training']
_epochs           = _tcfg['epochs']
_clip_norm        = _tcfg['gradient_clip_norm']            # required key
_two_stage        = _tcfg['two_stage']                     # required key
_warmup_epochs    = _two_stage['head_warmup_epochs']       # required key
_patience         = _tcfg['early_stopping_patience']       # required key

best_metric       = 0.0
best_state        = None
patience_counter  = 0
training_log      = []
backbones_unfrozen = False

print(f'\n{"="*80}')
print(f'[Train] Starting {_epochs} epochs | seed={CONFIG["project"]["seed"]}')
print(f'[Train] Stage 1 : head warmup for {_warmup_epochs} epochs (backbones frozen)')
if _two_stage['enabled']:
    print(f'[Train] Stage 2 : backbone fine-tune from epoch {_warmup_epochs+1}')
print(f'{"="*80}\n')
print(f'{"Epoch":>5} | {"Stage":>6} | {"Tr Loss":>8} | {"Tr Acc":>7} | '
      f'{"Va Loss":>8} | {"Va Acc":>7} | {"Va F1":>7} | {"Va PR-AUC":>10}')
print('-'*80)

for epoch in range(_epochs):

    # ── Stage 2 transition ────────────────────────────────────────────────────
    if _two_stage['enabled'] and epoch == _warmup_epochs and not backbones_unfrozen:
        print(f'\n{"="*80}')
        print(f'[Train] STAGE 2 — unfreezing backbones at epoch {epoch+1}')
        if _two_stage['unfreeze_cv']:
            unfreeze_backbone(model_raw, 'cv_extractor',  _tcfg['lr_backbone_cv'],  optimizer)
        if _two_stage['unfreeze_nlp']:
            unfreeze_backbone(model_raw, 'nlp_extractor', _tcfg['lr_backbone_nlp'], optimizer)

        # Gradient checkpointing saves VRAM on 4 GB cards
        model_raw.cv_extractor.vision_model.gradient_checkpointing_enable()
        model_raw.nlp_extractor.text_model.gradient_checkpointing_enable()
        print('[Train]   Gradient checkpointing enabled for Stage 2')

        # Update head LR to stage-2 value
        optimizer.param_groups[0]['lr'] = _tcfg['lr_head_stage2']

        # Rebuild scheduler for Stage 2 (respects config scheduler type)
        scheduler = build_scheduler(optimizer, CONFIG)

        # Reset early stopping counter for Stage 2
        patience_counter = 0
        backbones_unfrozen = True
        _n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'[Train]   Trainable params now: {_n_trainable:,}')
        print(f'{"="*80}\n')

    _stage = 'head' if not backbones_unfrozen else 'full'

    _tr = train_epoch(model, train_loader, criterion, optimizer, scaler, device, _clip_norm)
    _va = validate_epoch(model, val_loader, criterion, device)

    # ── Step scheduler (correct API per scheduler type) ───────────────────────
    if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
        scheduler.step(_va['pr_auc'])
    else:                               # CosineAnnealingWarmRestarts and others
        scheduler.step()

    training_log.append({
        'epoch':          epoch+1,
        'stage':          _stage,
        'train_loss':     _tr['loss'],    'train_accuracy': _tr['accuracy'], 'train_f1': _tr['f1'],
        'val_loss':       _va['loss'],    'val_accuracy':   _va['accuracy'],
        'val_f1':         _va['f1'],      'val_auc_roc':    _va['auc_roc'],  'val_pr_auc': _va['pr_auc'],
    })

    print(f'{epoch+1:>5} | {_stage:>6} | {_tr["loss"]:>8.4f} | {_tr["accuracy"]:>7.4f} | '
          f'{_va["loss"]:>8.4f} | {_va["accuracy"]:>7.4f} | {_va["f1"]:>7.4f} | {_va["pr_auc"]:>10.4f}')

    # ── Best model checkpoint ─────────────────────────────────────────────────
    if _va['pr_auc'] > best_metric:
        best_metric = _va['pr_auc']
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print(f'[Train]   -> New best  PR-AUC: {best_metric:.4f}  (checkpoint saved)')
    else:
        patience_counter += 1
        if patience_counter >= _patience:
            print(f'\n[Train] Early stopping after epoch {epoch+1} (patience={_patience})')
            break

print(f'\n[Train] Complete.  Best val PR-AUC: {best_metric:.4f}')



[Train] Starting 50 epochs | seed=42
[Train] Stage 1 : head warmup for 5 epochs (backbones frozen)
[Train] Stage 2 : backbone fine-tune from epoch 6

Epoch |  Stage |  Tr Loss |  Tr Acc |  Va Loss |  Va Acc |   Va F1 |  Va PR-AUC
--------------------------------------------------------------------------------


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve

if best_state is None:
    raise RuntimeError('[Eval] No checkpoint found — training may not have completed.')

model.load_state_dict(best_state)
model.eval()

@torch.no_grad()
def collect_predictions(mdl, loader):
    _probs, _labels = [], []
    for _b in loader:
        _logits = mdl(_b['image'].to(device), _b['input_ids'].to(device), _b['attention_mask'].to(device))
        _probs.extend(torch.sigmoid(_logits).cpu())
        _labels.extend(_b['label'])
    return torch.stack(_probs).numpy(), torch.stack(_labels).numpy()

# ── Threshold tuning on VALIDATION set (no test leakage) ─────────────────────
print('[Eval] Finding optimal threshold on validation set...')
_vp, _vl     = collect_predictions(model, val_loader)
_prec, _rec, _thr = precision_recall_curve(_vl, _vp)
_f1s         = 2 * _prec * _rec / (_prec + _rec + 1e-10)
_best_idx    = int(_f1s.argmax())
optimal_thr  = float(_thr[_best_idx]) if _best_idx < len(_thr) else 0.5
if optimal_thr < 0.2:
    print(f'[Eval] Note: threshold {optimal_thr:.4f} is very low — '
          'this can be normal for imbalanced data.')
print(f'[Eval] Optimal threshold : {optimal_thr:.4f}  (val F1: {_f1s[_best_idx]:.4f})')

# ── Test set evaluation ───────────────────────────────────────────────────────
print('[Eval] Evaluating on test set...')
_tp, _tl = collect_predictions(model, test_loader)

_preds_opt   = (_tp > optimal_thr).astype(int)
_preds_fixed = (_tp > 0.5).astype(int)

print(f'\n{"="*60}')
print('TEST SET RESULTS')
print(f'{"="*60}')
print(f'\nWith optimal threshold ({optimal_thr:.4f}):')
print(f'  Accuracy  : {(_preds_opt == _tl).mean():.4f}')
from sklearn.metrics import f1_score as _f1fn, roc_auc_score as _roc, average_precision_score as _ap
print(f'  F1        : {_f1fn(_tl, _preds_opt):.4f}')
print(f'  AUC-ROC   : {_roc(_tl, _tp):.4f}')
print(f'  PR-AUC    : {_ap(_tl, _tp):.4f}')
print(f'  Confusion :\n{confusion_matrix(_tl, _preds_opt).tolist()}')
print(f'\nWith fixed threshold (0.50):')
print(f'  Accuracy  : {(_preds_fixed == _tl).mean():.4f}')
print(f'  F1        : {_f1fn(_tl, _preds_fixed):.4f}')
print(f'  Confusion :\n{confusion_matrix(_tl, _preds_fixed).tolist()}')
print(f'{"="*60}')


In [ ]:
import pandas as pd

_model_path = CONFIG['paths']['best_model']
_log_path   = CONFIG['paths']['training_log']

if best_state is not None:
    torch.save(best_state, _model_path)
    print(f'[Save] Best model     -> {os.path.abspath(_model_path)}')
else:
    print('[Save] WARNING: no best_state found — model file not written.')

pd.DataFrame(training_log).to_csv(_log_path, index=False)
print(f'[Save] Training log   -> {os.path.abspath(_log_path)}')

# ── Colab: auto-download outputs ─────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files as _cf
    if os.path.exists(_model_path):
        _cf.download(_model_path)
    if os.path.exists(_log_path):
        _cf.download(_log_path)

print('[Save] Pipeline complete.')
